# 04 — Evaluate all variants & build the results table

Reproduces the in-dist vs cross-gen comparison table from the proposal.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ECS111FinalProject

## Run baselines

In [ ]:
!python -m src.baselines.tfidf_lr --out results/tfidf_lr.json
!python -m src.baselines.deberta_zero_shot --out results/deberta_zero_shot.json

## Evaluate fine-tuned variants across all rotations

In [ ]:
import json
from pathlib import Path
from src.evaluate import evaluate_checkpoint

CONFIGS = ['base', 'contrastive', 'adversarial', 'both']
ROTATIONS = [0, 1, 2, 3]
Path('results').mkdir(exist_ok=True)

all_results = {}
for config in CONFIGS:
    all_results[config] = {}
    for rot in ROTATIONS:
        ckpt_dir = f'checkpoints/{config}/rotation_{rot}'
        if not Path(f'{ckpt_dir}/best.pt').exists():
            print(f'missing ckpt: {ckpt_dir}'); continue
        res = evaluate_checkpoint(ckpt_dir, 'data/splits', rot)
        all_results[config][f'rotation_{rot}'] = res
        print(f'{config} r{rot}: indist F1 {res["test_indist"]["f1"]:.3f}, cross F1 {res["test_crossgen"]["f1"]:.3f}')

with open('results/finetuned.json', 'w') as f:
    json.dump(all_results, f, indent=2)

## Build the comparison table

In [ ]:
import json, pandas as pd
import numpy as np

def avg(results, split):
    f1s = [results[r][split]['f1'] for r in results if split in results[r]]
    aucs = [results[r][split]['auc'] for r in results if split in results[r]]
    return np.mean(f1s), np.mean(aucs)

rows = []
with open('results/tfidf_lr.json') as f: tfidf = json.load(f)
with open('results/deberta_zero_shot.json') as f: zero = json.load(f)
with open('results/finetuned.json') as f: ft = json.load(f)

in_f1, in_auc = avg(tfidf, 'test_indist'); cr_f1, cr_auc = avg(tfidf, 'test_crossgen')
rows.append(['TF-IDF + LR', in_f1, cr_f1, in_auc, cr_auc])

in_f1, in_auc = avg(zero, 'test_indist'); cr_f1, cr_auc = avg(zero, 'test_crossgen')
rows.append(['DeBERTa zero-shot', in_f1, cr_f1, in_auc, cr_auc])

for cfg in ['base', 'contrastive', 'adversarial', 'both']:
    in_f1, in_auc = avg(ft[cfg], 'test_indist'); cr_f1, cr_auc = avg(ft[cfg], 'test_crossgen')
    rows.append([f'DeBERTa-FT {cfg}', in_f1, cr_f1, in_auc, cr_auc])

df = pd.DataFrame(rows, columns=['Model', 'F1 in-dist', 'F1 cross-gen', 'AUC in-dist', 'AUC cross-gen'])
print(df.to_markdown(index=False, floatfmt='.3f'))
df.to_csv('results/summary.csv', index=False)

## Per-held-out-generator breakdown

Shows which generators are easiest/hardest to detect when unseen.

In [ ]:
GENERATORS = ['gpt5mini', 'granite', 'gemma', 'qwen']
rows = []
for cfg in ['base', 'contrastive', 'adversarial', 'both']:
    row = [cfg]
    for rot, gen in enumerate(GENERATORS):
        try:
            row.append(ft[cfg][f'rotation_{rot}']['test_crossgen']['f1'])
        except KeyError:
            row.append(float('nan'))
    rows.append(row)
perdf = pd.DataFrame(rows, columns=['variant'] + GENERATORS)
print(perdf.to_markdown(index=False, floatfmt='.3f'))

## In-dist vs cross-gen gap plot

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 5))
x = range(len(df))
ax.bar([i - 0.2 for i in x], df['F1 in-dist'], width=0.4, label='in-dist')
ax.bar([i + 0.2 for i in x], df['F1 cross-gen'], width=0.4, label='cross-gen')
ax.set_xticks(list(x)); ax.set_xticklabels(df['Model'], rotation=30, ha='right')
ax.set_ylabel('F1'); ax.legend(); ax.set_title('In-distribution vs Cross-Generator F1')
plt.tight_layout(); plt.savefig('results/gap_plot.png', dpi=150); plt.show()